In [ ]:
from collections import Counter, OrderedDict, defaultdict
from abc import ABC, abstractmethod
from typing import Dict, Any, Sequence, Tuple, Optional

import numpy as np

In [ ]:
CacheKey = Tuple[int, int, int]  # (vid, layer, tile)

class CachePolicy(ABC):
    @abstractmethod
    def get(self, key: CacheKey) -> Any: ...
    @abstractmethod
    def put(self, key: CacheKey, value: Any, size: int) -> Tuple[bool, list]: ...
    @abstractmethod
    def contains(self, key: CacheKey) -> bool: ...
    @abstractmethod
    def remove(self, key: CacheKey) -> bool: ...
    @abstractmethod
    def clear(self) -> None: ...
    @abstractmethod
    def keys(self): ...
    @abstractmethod
    def stats(self) -> Dict[str, Any]: ...

In [ ]:
class LruPolicy(CachePolicy):
    def __init__(self, max_size: int):
        self.cur_size = 0
        self.max_size = max_size
        self.cache = {}             # (vid, layer, tile) -> (value, size)
        self.access_order = []      # keys in order of access (oldest first)
    
    def get(self, key):
        if key not in self.cache:
            return None

        self.access_order.remove(key)
        self.access_order.append(key)
        
        return self.cache[key][0]
    
    def put(self, key, value, size: int):
        evicted = []

        if key in self.cache:
            old_size = self.cache[key][1]
            self.cur_size -= old_size
            self.access_order.remove(key)
            del self.cache[key]

        while self.cur_size + size > self.max_size and self.access_order:
            lru_key = self.access_order.pop(0)  # Remove oldest (least recently used)
            lru_size = self.cache[lru_key][1]
            self.cur_size -= lru_size
            del self.cache[lru_key]
            evicted.append(lru_key)

        # Add new item if there's space
        if self.cur_size + size <= self.max_size:
            self.cache[key] = (value, size)
            self.access_order.append(key)
            self.cur_size += size

        return evicted
    
    def contains(self, key) -> bool:
        return key in self.cache
    
    def remove(self, key):
        if key not in self.cache:
            return False
        
        size = self.cache[key][1]
        self.cur_size -= size
        self.access_order.remove(key)
        del self.cache[key]

        return True
    
    def clear(self):
        self.cache.clear()
        self.access_order.clear()
        self.cur_size = 0
    
    def get_stats(self) -> Dict[str, Any]:
        return {
            'size': self.cur_size,
            'max_size': self.max_size,
            'num_items': len(self.cache),
            'utilization': self.cur_size / self.max_size if self.max_size > 0 else 0
        }
    
    def keys(self):
        return self.cache.keys()
    
    def stats(self) -> Dict[str, Any]:
        return {
            'size': self.cur_size,
            'max_size': self.max_size,
            'num_items': len(self.cache),
            'utilization': self.cur_size / self.max_size if self.max_size > 0 else 0
        }

In [ ]:
class SvcLruPolicy(LruPolicy):
    def __init__(self, max_size: int):
        super().__init__(max_size)

    def put(self, key, value, size: int):
        evicted = []
        if key in self.cache:
            old_size = self.cache[key][1]
            self.cur_size -= old_size
            self.access_order.remove(key)
            del self.cache[key]
        
        while self.cur_size + size > self.max_size and self.access_order:
            victim_key = None

            for k in self.access_order:
                if len(k) > 1 and k[1] == 1:
                    victim_key = k
                    break

            if victim_key is None:
                victim_key = self.access_order[0]
            
            victim_size = self.cache[victim_key][1]
            self.cur_size -= victim_size
            self.access_order.remove(victim_key)
            del self.cache[victim_key]
            evicted.append(victim_key)            

        if self.cur_size + size <= self.max_size:
            self.cache[key] = (value, size)
            self.access_order.append(key)
            self.cur_size += size
        
        return evicted

In [ ]:
class LfuPolicy(CachePolicy):
    def __init__(self, max_size: int):
        self.cur_size = 0
        self.max_size = max_size
        self.cache: Dict[Tuple[int, int, int, int], Tuple[Any, int]] = {}
        self.freq: Dict[Tuple[int, int, int, int], int] = {}
        self.freq_to_keys: Dict[int, OrderedDict] = defaultdict(OrderedDict)
        self.min_freq = 0

    def _increment_freq(self, key):
        f = self.freq.get(key, 0)
        if f in self.freq_to_keys and key in self.freq_to_keys[f]:
            # Remove from current frequency bucket
            self.freq_to_keys[f].pop(key, None)
            if not self.freq_to_keys[f]:
                del self.freq_to_keys[f]
                if self.min_freq == f:
                    self.min_freq = f + 1
        # Add to next frequency bucket
        nf = f + 1
        self.freq[key] = nf
        self.freq_to_keys[nf][key] = None

    def get(self, key):
        if key not in self.cache:
            return None
        self._increment_freq(key)
        return self.cache[key][0]

    def _evict_one(self):
        if not self.freq_to_keys:
            return None
        # Ensure min_freq points to an existing bucket
        if self.min_freq not in self.freq_to_keys:
            if self.freq_to_keys:
                self.min_freq = min(self.freq_to_keys.keys())
            else:
                return None
        # Evict least-recently used within the minimum frequency bucket
        victim_key, _ = self.freq_to_keys[self.min_freq].popitem(last=False)
        if not self.freq_to_keys[self.min_freq]:
            del self.freq_to_keys[self.min_freq]
        victim_size = self.cache[victim_key][1]
        self.cur_size -= victim_size
        del self.cache[victim_key]
        self.freq.pop(victim_key, None)
        return victim_key

    def put(self, key, value, size: int):
        evicted = []

        if key in self.cache:
            # Adjust size; treat as an access as well
            old_size = self.cache[key][1]
            self.cur_size -= old_size
            # Update payload before incrementing frequency
            self.cache[key] = (value, size)
            # Bring key to higher freq (counts as access)
            self._increment_freq(key)
        else:
            # New key starts with freq=1; set up before capacity adjustments
            self.cache[key] = (value, size)
            self.freq[key] = 0  # will become 1 after increment
            self.min_freq = 1 if self.min_freq in (0, 1) else min(self.min_freq, 1)
            self._increment_freq(key)

        # Evict until it fits
        while self.cur_size + size > self.max_size:
            victim = self._evict_one()
            if victim is None:
                break
            evicted.append(victim)

        # Add size if it fits; otherwise revert new item
        if self.cur_size + size <= self.max_size:
            self.cur_size += size
        else:
            # Could not fit: remove the just-updated/inserted key
            # Clean up from structures
            f = self.freq.pop(key, None)
            if f is not None and f in self.freq_to_keys and key in self.freq_to_keys[f]:
                self.freq_to_keys[f].pop(key, None)
                if not self.freq_to_keys[f]:
                    del self.freq_to_keys[f]
            # Remove from cache
            if key in self.cache:
                del self.cache[key]
            # Recompute min_freq
            if self.freq_to_keys:
                self.min_freq = min(self.freq_to_keys.keys())
            else:
                self.min_freq = 0

        return evicted

    def contains(self, key) -> bool:
        return key in self.cache

    def remove(self, key):
        if key not in self.cache:
            return False
        size = self.cache[key][1]
        self.cur_size -= size
        del self.cache[key]
        f = self.freq.pop(key, None)
        if f is not None and f in self.freq_to_keys:
            self.freq_to_keys[f].pop(key, None)
            if not self.freq_to_keys[f]:
                del self.freq_to_keys[f]
        if self.freq_to_keys:
            self.min_freq = min(self.freq_to_keys.keys())
        else:
            self.min_freq = 0
        return True

    def clear(self):
        self.cache.clear()
        self.freq.clear()
        self.freq_to_keys.clear()
        self.cur_size = 0
        self.min_freq = 0

    def keys(self):
        return self.cache.keys()

    def get_stats(self) -> Dict[str, Any]:
        return {
            'size': self.cur_size,
            'max_size': self.max_size,
            'num_items': len(self.cache),
            'utilization': self.cur_size / self.max_size if self.max_size > 0 else 0
        }

    def stats(self) -> Dict[str, Any]:
        return {
            'size': self.cur_size,
            'max_size': self.max_size,
            'num_items': len(self.cache),
            'utilization': self.cur_size / self.max_size if self.max_size > 0 else 0
        }

In [ ]:
class CacheUnitMapper:
    """
    Maps MB-based cache to logical cache units (paper abstraction)
    """
    def __init__(
        self,
        cache_capacity_mb,
        num_gops,
        num_tiles,
        viewport_tiles,
        base_tile_mb,
        enh_tile_mb
    ):
        self.unit_mb = (
            num_gops * num_tiles * base_tile_mb +
            num_gops * viewport_tiles * enh_tile_mb
        )

        self.max_units = int(cache_capacity_mb // self.unit_mb)

        self.viewport_tiles = viewport_tiles
        
    def units_from_mb(self, used_mb):
        return used_mb / self.unit_mb

    def can_add_unit(self, current_units):
        return current_units < self.max_units

In [ ]:
class CacheEngineEnv:
    def __init__(
        self,
        n_users: int = 1000,
        n_tiles: int = 16,
        n_layers: int = 2,
        n_gops: int = 60,
        n_videos: int = 100,
        cache_capacity: float = 100e6,  # capacity in bytes
        policy: CachePolicy | None = None,
        unit_mapper: CacheUnitMapper | None = None,
    ):
        self.max_capacity = cache_capacity
        self.tile_size_bytes = {
            0: 2e6 / n_tiles,   # base layer tile size in bytes
            1: 15e6 / n_tiles   # enhancement layer tile size in bytes
        }
        self.n_users = n_users
        self.n_layers = n_layers
        self.n_videos = n_videos
        self.n_gops = n_gops
        self.n_tiles = n_tiles

        # Initialize LRU policy
        self.policy = policy or LruPolicy(max_size=int(cache_capacity))

        self.cache_bitmap = np.zeros(
            (self.n_videos, self.n_layers, self.n_tiles, self.n_gops), dtype=np.int8
        )

        self.content_popularity = np.zeros((n_videos,), dtype=np.float32)
        self.user_visited = np.zeros((n_users, n_videos), dtype=bool)
        
        # Initialize Cache Unit Mapper
        self.unit_mapper = unit_mapper

    def _cache_tile(self, vid_idx, layer_idx, tile_idx, gop_idx):
        key = (vid_idx, layer_idx, tile_idx, gop_idx)
        tile_size = self.tile_size_bytes[layer_idx]

        evicted = self.policy.put(key, None, int(tile_size))

        # Update cache bitmap
        self.cache_bitmap[vid_idx, layer_idx, tile_idx, gop_idx] = 1
        for e_vid, e_layer, e_tile, e_gop in evicted:
            self.cache_bitmap[e_vid, e_layer, e_tile, e_gop] = 0

    def _clear_cache(self):
        self.policy.clear()

    def update_content_popularity(self, req: Dict[str, Any]):
        user = req['u']
        vid = req['video']

        if self.user_visited[user, vid]:
            return

        self.content_popularity[vid] += 1
        self.user_visited[user, vid] = True

    def get_content_popularity(self):
        return self.content_popularity

    def get_video_popularity_norm(self, vid_id: int) -> float:
        total_requests = np.sum(self.content_popularity)
        
        # Avoid division by zero at the start of the episode
        if total_requests == 0:
            return 0.0
            
        # Return probability: P(v) = count(v) / total_count
        return self.content_popularity[vid_id] / total_requests

    def get_tile(self, vid, layer, tile_idx, gop_idx):
        key = (vid, layer, tile_idx, gop_idx)
        if self.policy.contains(key):
            self.policy.get(key)  # Update access order
            return True
        return False
    
    def get_current_capacity(self):
        return self.policy.cur_size
    
    def get_cache_bitmap(self):
        return self.cache_bitmap
    
    def _cache_base_layer(self, vid_idx, layer_idx):
        key = (vid_idx, layer_idx, slice(None), slice(None))
        tile_size = self.tile_size_bytes[layer_idx] * self.n_tiles * self.n_gops

        evicted = self.policy.put(key, None, int(tile_size))
        
        # Update cache bitmap
        self.cache_bitmap[vid_idx, layer_idx, :, :] = 1
        for key in evicted:
            self.cache_bitmap[key] = 0

    def prefetching(self, action):

        if action is None or action['gop'] >= self.n_gops:
            return self.get_cache_bitmap()

        gop = action['gop']
        vid = action['video']
        tiles = action['tiles']
        base_req_init = action['base_req_init']

        if base_req_init:
            self._cache_base_layer(vid, 0)

        if not self.is_video_cached(vid, 0):
            return self.get_cache_bitmap()

        for tile_idx, tile in enumerate(tiles):
            if tile == 1:
                self._cache_tile(vid, 1, tile_idx, gop)

        return self.get_cache_bitmap()

    def get_slots(self):
        """
        Returns the list of currently cached Video IDs (Base Layer).
        Used by the Agent to see which videos are present.
        """
        cached_slots = set()
        for key in self.policy.keys():
            vid, layer, tile_idx, gop_idx = key
            if layer == 0:  # Base layer
                cached_slots.add(vid)
        return list(cached_slots)

    def get_cached_tiles(self):
        """
        Returns a list of lists, where each inner list contains the Tile IDs 
        cached in high quality for the corresponding video slot.
        """
        cached_tiles = defaultdict(list)
        for key in self.policy.keys():
            vid, layer, tile_idx, gop_idx = key
            if layer == 1:  # Enhancement layer
                cached_tiles[vid].append(tile_idx)
        
        # Convert defaultdict to regular dict and ensure all vids are represented
        result = []
        for vid in range(self.n_videos):
            result.append(cached_tiles.get(vid, []))
        
        return result

    def is_video_cached(self, vid_id: int, layer_id: int) -> bool:
        """
        Checks if any tile of the given video ID is cached in the base layer.
        """
        return np.any(self.cache_bitmap[vid_id, layer_id, :, :] == 1)

    def is_tile_cached(self, vid_id: int, layer_id: int, tile_id: int, gop_id: int) -> bool:
        return self.cache_bitmap[vid_id, layer_id, tile_id, gop_id] == 1

    def evict_video(self):
        video_id = self.select_eviction_candidate()
        tiles = self.remove_video_tiles(video_id)

        freed_mb = sum(t.size_mb for t in tiles)
        self.used_mb -= freed_mb

    def add_tile(self, video_id, gop_id, layer_id, tile_id, size_mb):
        key = (video_id, layer_id, tile_id, gop_id)
        tile_size = size_mb

        evicted = self.policy.put(key, None, int(tile_size * 1e6))

        for e_vid, e_layer, e_tile, e_gop in evicted:
            self.cache_bitmap[e_vid, e_layer, e_tile, e_gop] = 0

        self.cache_bitmap[video_id, layer_id, tile_id, gop_id] = 1
        self.used_mb += tile_size

    def cache_new_video(self, video_id, gop_id=0, layer=0, tile_id=None):
        if layer == 0:
            self.cache_bitmap[video_id, 0, :, :] = 1
        elif layer == 1:
            self.cache_bitmap[video_id, 1, tile_id, gop_id] = 1

    def get_viewport_tile_budget(self):
        return 4 # Fixed to 4 as per paper

    def reset(self, **kwargs):
        self._clear_cache()

        self.content_popularity = np.zeros((self.n_videos,), dtype=np.float32)
        self.user_visited = np.zeros((self.n_users, self.n_videos), dtype=bool)

        self.cache_bitmap = np.zeros(
            (self.n_videos, self.n_layers, self.n_tiles, self.n_gops), dtype=np.int8
        )

        ### Randomly pre-fill cache ###
        # rng = np.random.default_rng()
        # keys = [(v, l, t) for v in range(self.n_videos)
        #                  for l in range(self.n_layers)
        #                  for t in range(self.n_tiles)]
        # rng.shuffle(keys)

        # for vid, layer, tile_id in keys:
        #     tile_size = int(self.tile_size_bytes[layer])
        #     if self.policy.cur_size + tile_size > self.max_capacity:
        #         break

        #     # Random GOP index for each tile
        #     gop_idx = rng.integers(0, self.n_gops)
        #     if not self.policy.contains((vid, layer, tile_id, gop_idx)):
        #         self._cache_tile(vid, layer, tile_id, gop_idx)
        ### End Randomly pre-fill cache ###

        return None, {"cache": self.get_cache_bitmap()}


In [ ]:
if __name__ == "__main__":
    # Test configuration
    n_tiles = 4
    n_layers = 2
    n_videos = 10
    cache_capacity = 10e6  # 10 MB
    
    print("=" * 5, "Cache Storage Test with LRU Policy (Computed Matrix)", "=" * 5)
    
    # Initialize cache environment with LRU
    cache = CacheEngineEnv(
        n_tiles=n_tiles * n_tiles,
        n_layers=n_layers,
        n_videos=n_videos,
        cache_capacity=cache_capacity
    )
    
    cache.reset()

In [ ]:
# Test LRU Policy with tuple keys (video, layer, tile)
if __name__ == "__main__":
    print("=" * 5, "LRU Cache Policy Test", "=" * 5)
    
    # Create LRU cache with 10 MB capacity
    lru = LruPolicy(max_size=10 * 1024 * 1024)  # 10 MB
    
    print(f"Initial state: {lru.get_stats()}\n")
    
    # Add some items using tuple keys (video, layer, tile)
    print("Adding items:")
    evicted = lru.put(
        (0, 0, 0), 
        "data_v0_l0_t0", 
        2 * 1024 * 1024
    )  # 2 MB
    print(f"  Added (0,0,0) (2 MB), evicted: {evicted}")
    
    evicted = lru.put(
        (1, 0, 0), 
        "data_v1_l0_t0", 
        3 * 1024 * 1024
    )  # 3 MB
    print(f"  Added (1,0,0) (3 MB), evicted: {evicted}")
    
    evicted = lru.put(
        (2, 0, 0), 
        "data_v2_l0_t0", 
        4 * 1024 * 1024
    )  # 4 MB
    print(f"  Added (2,0,0) (4 MB), evicted: {evicted}")
    
    print(f"\nCurrent state: {lru.get_stats()}")
    print(f"Access order (oldest→newest): {lru.access_order}\n")
    
    # Access tile (0,0,0) (moves it to most recent)
    print("Accessing (0,0,0)...")
    data = lru.get((0, 0, 0))
    print(f"  Retrieved: {data}")
    print(f"  New access order: {lru.access_order}\n")
    
    # Add item that requires eviction
    print("Adding large item (3 MB) - should evict LRU item...")
    evicted = lru.put((3, 0, 0), "data_v3_l0_t0", 3 * 1024 * 1024)
    print(f"  Evicted: {evicted}")
    print(f"  Current access order: {lru.access_order}")
    print(f"  State: {lru.get_stats()}\n")
    
    # Test contains
    print("Testing contains:")
    for key in [(0, 0, 0), (1, 0, 0), (2, 0, 0), (3, 0, 0)]:
        print(f"  {key}: {lru.contains(key)}")
    
    print("\n" + "=" * 60)

In [ ]:
if __name__ == "__main__":
    # Test configuration
    n_tiles = 4
    n_layers = 2
    n_videos = 10
    cache_capacity = 10e6  # 10 MB
    
    print("=" * 5, "Cache Storage Test with LRU Policy (Computed Matrix)", "=" * 5)
    
    # Initialize cache environment with LRU
    cache = CacheEngineEnv(
        n_tiles=n_tiles * n_tiles,
        n_layers=n_layers,
        n_videos=n_videos,
        cache_capacity=cache_capacity
    )
    
    print(f"Configuration:")
    print(f"  Grid size: {n_tiles}x{n_tiles} = {n_tiles*n_tiles} tiles")
    print(f"  Layers: {n_layers}")
    print(f"  Videos: {n_videos}")
    print(f"  Capacity: {cache_capacity/1e6:.1f} MB")
    print(f"  Base layer tile size: {cache.tile_size_bytes[0]/1e6:.3f} MB")
    print(f"  Enhancement tile size: {cache.tile_size_bytes[1]/1e6:.3f} MB")
    print(f"  Using LRU: {cache.policy is not None}")
    print()
    
    # Create sample cache actions (prefetch decisions)
    action = [
        {
            'video': 0,
            'gop': 0,
            'tiles': np.array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
        },
        {
            'video': 1,
            'gop': 2,
            'tiles': np.array([0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
        },
        {
            'video': 0,
            'gop': 1,
            'tiles': np.array([1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
        },
    ]
    
    print(f"Processing {len(action)} cache actions...")
    print(f"Video requests: {[act['video'] for act in action]}\n")
    
    # Process cache prefetching
    cache_matrix = cache.prefetching(action)
    
    print("Cache Results:")
    current_size = cache.get_current_capacity()
    print(f"  Final capacity used: {current_size/1e6:.2f} MB / {cache.max_capacity/1e6:.1f} MB")
    print(f"  Utilization: {(current_size/cache.max_capacity)*100:.1f}%")
    
    if cache.policy:
        stats = cache.policy.get_stats()
        print(f"  LRU stats: {stats['num_items']} items, {stats['utilization']:.1%} full")
    print()
    
    # Show cache content per video
    print("Cached tiles by video:")
    for vid_idx in range(n_videos):
        base_cached = np.sum(cache_matrix[vid_idx, 0, :, :])
        enh_cached = np.sum(cache_matrix[vid_idx, 1, :, :])
        if base_cached > 0 or enh_cached > 0:
            print(f"  Video {vid_idx}: Base={base_cached}/{n_tiles*n_tiles}, Enh={enh_cached}/{n_tiles*n_tiles}")
            if base_cached > 0:
                print(f"    Base layer tiles: {np.where(cache_matrix[vid_idx, 0, :] == 1)[0].tolist()}")
            if enh_cached > 0:
                print(f"    Enh layer tiles:  {np.where(cache_matrix[vid_idx, 1, :] == 1)[0].tolist()}")
    print()
    
    print("Testing tile access (updates LRU):")
    print(f"  Accessing tile (0, 0, 5): {cache.get_tile(0, 0, 5, 0)}")
    print(f"  Accessing tile (1, 0, 3): {cache.get_tile(1, 0, 3, 2)}")
    print(f"  Accessing non-cached (5, 0, 0): {cache.get_tile(5, 0, 0, 0)}")